# 09 modelos 20 y 21


## Modelo 20 — Regularización y margen mínimo de rentabilidad

**Qué probamos.** Mantuvimos las características de la V7 y comparamos `C=1`, `C=0.1` y `C=0.01`. También exigimos distintos mínimos de ganancia esperada por dólar: 0, 0,02, 0,05 y 0,10.

**Qué pasó.** `C=0.1` siguió siendo la mejor configuración de las tres en la validación. Con `C=0.01` perdíamos USD 342,59 de promedio y con `C=1` perdíamos USD 2.081,30. Aumentar el umbral no aportó una mejora.

**Qué decidimos.** No cambiar la V7. Como el presupuesto ya se completaba con préstamos de valor esperado relativamente alto, esos umbrales cambiaban poco o nada de la selección.

In [ ]:

import numpy as np
import pandas as pd

# comprobamos los datos del modelo anterior
columnas_m20 = list(m19_alternativas)

assert len(m19_train) == len(train)
assert len(m19_test) == len(test)
assert m19_train["ID_Solicitud"].is_unique
assert m19_test["ID_Solicitud"].is_unique

# seleccionamos prestamos segun la rentabilidad esperada
def seleccionar_cartera_m20(datos, probabilidades, presupuesto, umbral=0):
    cartera = datos[
        ["ID_Solicitud", "Monto_Solicitado", "Tasa_Interes"]
    ].copy().reset_index(drop=True)

    cartera["Prob_Pago"] = np.asarray(probabilidades)

    cartera["EV_Dolar"] = (
        cartera["Prob_Pago"] * (1 + cartera["Tasa_Interes"]) - 1
    )

    cartera = cartera.sort_values(
        ["EV_Dolar", "ID_Solicitud"],
        ascending=[False, True]
    ).reset_index(drop=True)

    cartera["Aprobar"] = 0
    capital = 0.0

    # recorremos los prestamos sin superar el presupuesto
    for i, fila in cartera.iterrows():
        monto = fila["Monto_Solicitado"]

        if fila["EV_Dolar"] <= umbral:
            break

        if capital + monto <= presupuesto:
            cartera.loc[i, "Aprobar"] = 1
            capital += monto

    return cartera


# calculamos la ganancia real de una cartera de validacion
def medir_cartera_m20(cartera, etiquetas, presupuesto):
    datos = cartera.merge(
        etiquetas[["ID_Solicitud", "Estado_Final"]],
        on="ID_Solicitud",
        validate="one_to_one"
    )

    aprobados = datos[datos["Aprobar"] == 1]

    capital = float(aprobados["Monto_Solicitado"].sum())

    ganancia = float(np.where(
        aprobados["Estado_Final"] == 1,
        aprobados["Monto_Solicitado"] * aprobados["Tasa_Interes"],
        -aprobados["Monto_Solicitado"]
    ).sum())

    return {
        "Ganancia": ganancia,
        "ROI_%": 100 * ganancia / capital if capital else 0,
        "Capital": capital,
        "Uso_Presupuesto_%": (
            100 * capital / presupuesto if presupuesto else 0
        ),
        "Aprobados": len(aprobados),
        "Defaults": int(
            (aprobados["Estado_Final"] == 0).sum()
        ),
        "Sesgo_Aprobados": (
            float(
                aprobados["Prob_Pago"].mean()
                - aprobados["Estado_Final"].mean()
            )
            if len(aprobados) else np.nan
        ),
        "Menor_EV_Aprobado": (
            float(aprobados["EV_Dolar"].min())
            if len(aprobados) else np.nan
        )
    }

print("Funciones preparadas correctamente")


In [ ]:

# reconstruimos la regresion anterior
modelo_previo_m20 = crear_modelo_m16("Logistica")

modelo_previo_m20.fit(
    train_fe[columnas_m16],
    train_fe["Estado_Final"]
)

clases_previas = list(
    modelo_previo_m20.named_steps["modelo"].classes_
)

prob_previa_m20 = modelo_previo_m20.predict_proba(
    test_fe[columnas_m16]
)[:, clases_previas.index(1)]

cartera_previa_m20 = seleccionar_cartera_m20(
    test_fe, prob_previa_m20, 2_000_000
)

# entrenamos nuevamente la configuracion de la v7
train_v7_m20 = m19_train[
    m19_train["Monto_Solicitado"] <= m19_max_test
]

modelo_v7_m20 = crear_logistica_m19(
    columnas_m20, C=0.1
)

modelo_v7_m20.fit(
    train_v7_m20[columnas_m20],
    train_v7_m20["Estado_Final"]
)

clases_v7_m20 = list(
    modelo_v7_m20.named_steps["modelo"].classes_
)

prob_v7_m20 = modelo_v7_m20.predict_proba(
    m19_test[columnas_m20]
)[:, clases_v7_m20.index(1)]

cartera_v7_m20 = seleccionar_cartera_m20(
    m19_test, prob_v7_m20, 2_000_000
)

# comparamos las decisiones de las dos versiones
comparacion_carteras_m20 = cartera_previa_m20[
    ["ID_Solicitud", "Aprobar"]
].rename(columns={
    "Aprobar": "Previa"
}).merge(
    cartera_v7_m20[
        ["ID_Solicitud", "Monto_Solicitado", "Aprobar"]
    ].rename(columns={"Aprobar": "V7"}),
    on="ID_Solicitud",
    validate="one_to_one"
)

comparacion_carteras_m20["Grupo"] = np.select(
    [
        (comparacion_carteras_m20["Previa"] == 1) &
        (comparacion_carteras_m20["V7"] == 1),

        (comparacion_carteras_m20["Previa"] == 1) &
        (comparacion_carteras_m20["V7"] == 0),

        (comparacion_carteras_m20["Previa"] == 0) &
        (comparacion_carteras_m20["V7"] == 1)
    ],
    ["Ambas", "Solo anterior", "Solo V7"],
    default="Ninguna"
)

print("CAMBIOS EN LAS DECISIONES")

display(
    comparacion_carteras_m20.groupby("Grupo").agg(
        Cantidad=("ID_Solicitud", "count"),
        Capital=("Monto_Solicitado", "sum")
    ).round(2)
)

# revisamos las carteras aprobadas
for nombre, cartera in [
    ("Regresion anterior", cartera_previa_m20),
    ("V7", cartera_v7_m20)
]:
    aprobados = cartera[cartera["Aprobar"] == 1]

    print("\n", nombre)
    print("Aprobados:", len(aprobados))
    print("Capital:", aprobados["Monto_Solicitado"].sum())
    print("Monto promedio:", round(
        aprobados["Monto_Solicitado"].mean(), 2
    ))
    print("Probabilidad promedio:", round(
        aprobados["Prob_Pago"].mean(), 4
    ))
    print("Menor EV/dolar aprobado:", round(
        aprobados["EV_Dolar"].min(), 4
    ))
    print("Ganancia esperada:", round(
        (
            aprobados["EV_Dolar"] *
            aprobados["Monto_Solicitado"]
        ).sum(), 2
    ))


In [ ]:

m20_valores_c = [1.0, 0.1, 0.01]
m20_umbrales = [0.00, 0.02, 0.05, 0.10]

m20_resultados = []

# validamos los tres meses con las mismas condiciones
for mes in ["2025-10", "2025-11", "2025-12"]:
    inicio = pd.Timestamp(mes + "-01")
    fin = inicio + pd.offsets.MonthBegin(1)

    anteriores = m19_train[
        m19_train["Fecha_dt"] < inicio
    ]

    entrenamiento = anteriores[
        anteriores["Monto_Solicitado"] <= m19_max_test
    ]

    usuarios_vistos = set(anteriores["ID_Usuario"])

    validacion = m19_train[
        (m19_train["Fecha_dt"] >= inicio) &
        (m19_train["Fecha_dt"] < fin) &
        (m19_train["Monto_Solicitado"] <= m19_max_test) &
        (~m19_train["ID_Usuario"].isin(usuarios_vistos))
    ].copy()

    assert len(entrenamiento) > 0
    assert len(validacion) >= 30

    presupuesto = (
        validacion["Monto_Solicitado"].sum() *
        m19_proporcion
    )

    # entrenamos cada version sobre los mismos datos
    for valor_c in m20_valores_c:
        modelo = crear_logistica_m19(
            columnas_m20, C=valor_c
        )

        modelo.fit(
            entrenamiento[columnas_m20],
            entrenamiento["Estado_Final"]
        )

        clases = list(
            modelo.named_steps["modelo"].classes_
        )

        probabilidades = modelo.predict_proba(
            validacion[columnas_m20]
        )[:, clases.index(1)]

        # probamos distintos margenes de rentabilidad
        for umbral in m20_umbrales:
            cartera = seleccionar_cartera_m20(
                validacion,
                probabilidades,
                presupuesto,
                umbral
            )

            metricas = medir_cartera_m20(
                cartera, validacion, presupuesto
            )

            m20_resultados.append({
                "Mes": mes,
                "C": valor_c,
                "Umbral": umbral,
                "Candidatos": len(validacion),
                **metricas
            })

    print("Terminado:", mes)

m20_tabla = pd.DataFrame(m20_resultados)

print("Evaluaciones realizadas:", len(m20_tabla))


In [ ]:

# tomamos la configuracion de la v7 como referencia
referencia_m20 = m20_tabla[
    (m20_tabla["C"] == 0.1) &
    (m20_tabla["Umbral"] == 0.0)
].set_index("Mes")["Ganancia"]

m20_tabla["Diferencia_V7"] = (
    m20_tabla["Ganancia"] -
    m20_tabla["Mes"].map(referencia_m20)
)

# resumimos la ganancia de cada configuracion
m20_resumen = m20_tabla.groupby(
    ["C", "Umbral"]
).agg(
    Ganancia_Promedio=("Ganancia", "mean"),
    ROI_Promedio=("ROI_%", "mean"),
    Peor_ROI=("ROI_%", "min"),
    Diferencia_V7=("Diferencia_V7", "mean"),
    Meses_Mejora=(
        "Diferencia_V7",
        lambda valores: int((valores > 0.01).sum())
    ),
    Defaults_Promedio=("Defaults", "mean"),
    Aprobados_Promedio=("Aprobados", "mean"),
    Uso_Presupuesto=("Uso_Presupuesto_%", "mean"),
    Menor_EV_Aprobado=("Menor_EV_Aprobado", "mean")
).reset_index()

m20_resumen = m20_resumen.sort_values(
    ["Ganancia_Promedio", "Peor_ROI"],
    ascending=False
)

print("RESULTADOS GENERALES")
display(m20_resumen.round(3))

# comparamos las diferencias mensuales contra la v7
print("\nDIFERENCIAS MENSUALES CONTRA V7")

display(
    m20_tabla.pivot(
        index=["C", "Umbral"],
        columns="Mes",
        values="Diferencia_V7"
    ).round(2)
)

# revisamos las configuraciones sin margen adicional
print("\nREGULARIZACION SIN CAMBIAR LA SELECCION")

display(
    m20_resumen[
        m20_resumen["Umbral"] == 0
    ].round(3)
)

# mostramos el detalle de la mejor configuracion local
mejor_m20 = m20_resumen.iloc[0]

print("\nMEJOR CONFIGURACION LOCAL")
print("C:", mejor_m20["C"])
print("Umbral:", mejor_m20["Umbral"])
print("Ganancia promedio:", round(
    mejor_m20["Ganancia_Promedio"], 2
))
print("Diferencia contra V7:", round(
    mejor_m20["Diferencia_V7"], 2
))
print("Meses con mejora:", int(mejor_m20["Meses_Mejora"]))


## 8. Pruebas posteriores a la V7

Después de conseguir USD 130.062 en el leaderboard público seguimos revisando algunas ideas. Estos experimentos nos sirvieron para analizar el comportamiento de la V7, pero ninguno aportó una mejora suficientemente consistente como para reemplazarla.

### Modelo 21 — Historial bancario de 12 meses

**Qué probamos.** Recalculamos las mismas características bancarias usando una ventana fija de los 12 meses anteriores a cada solicitud. Mantuvimos la regresión logística, `C=0.1` y la selección por ganancia esperada por dólar.

**Qué pasó.** En la validación de junio a diciembre de 2025, la ventana de 12 meses ganó en 3 de los 7 meses. El promedio pasó de USD 281.024,23 a USD 281.232,96, una diferencia de solo USD 208,74. En las muestras de bootstrap tuvo peores resultados que la V7.

**Qué decidimos.** Mantener el historial utilizado por la V7. No había una ventaja estable que justificara cambiarlo.

In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# configuracion del experimento
m21_columnas = list(m19_alternativas)

m21_meses = [
    "2025-06", "2025-07", "2025-08",
    "2025-09", "2025-10", "2025-11", "2025-12"
]

# construimos las mismas variables usando solo los 12 meses anteriores
def preparar_datos_m21(solicitudes, banco):
    sol = solicitudes.copy()
    ban = banco.copy()

    sol["Fecha_dt"] = pd.to_datetime(sol["Fecha_Solicitud"])
    ban["Mes_dt"] = pd.to_datetime(ban["Mes_Referencia"])

    sol["_Mes_Solicitud"] = (
        sol["Fecha_dt"].dt.year * 12 +
        sol["Fecha_dt"].dt.month
    )

    ban["_Mes_Banco"] = (
        ban["Mes_dt"].dt.year * 12 +
        ban["Mes_dt"].dt.month
    )

    # vinculamos cada solicitud con su historial bancario
    historial = sol[
        ["ID_Solicitud", "ID_Usuario", "_Mes_Solicitud"]
    ].merge(
        ban[
            [
                "ID_Usuario", "_Mes_Banco",
                "Saldo_Promedio_Mensual",
                "Dias_Atraso_Otras_Deudas"
            ]
        ],
        on="ID_Usuario",
        how="left"
    )

    # descartamos meses futuros, el actual y registros muy antiguos
    distancia = (
        historial["_Mes_Solicitud"] -
        historial["_Mes_Banco"]
    )

    historial = historial[
        (distancia >= 1) & (distancia <= 12)
    ].copy()

    historial = historial.sort_values(
        ["ID_Solicitud", "_Mes_Banco"],
        kind="mergesort"
    )

    # calculamos las estadisticas generales del historial
    generales = historial.groupby("ID_Solicitud").agg(
        Saldo_Promedio=("Saldo_Promedio_Mensual", "mean"),
        Saldo_Minimo=("Saldo_Promedio_Mensual", "min"),
        Atraso_Maximo=("Dias_Atraso_Otras_Deudas", "max"),
        Atraso_Promedio=("Dias_Atraso_Otras_Deudas", "mean"),
        Meses_Observados=("_Mes_Banco", "count"),
        Meses_Con_Atraso=(
            "Dias_Atraso_Otras_Deudas",
            lambda x: (x > 0).sum()
        ),
        Meses_Atraso_30=(
            "Dias_Atraso_Otras_Deudas",
            lambda x: (x >= 30).sum()
        )
    ).reset_index()

    # usamos los tres registros mas recientes dentro de la ventana
    recientes = (
        historial.groupby("ID_Solicitud").tail(3)
        .groupby("ID_Solicitud")
        .agg(
            Saldo_Ultimos_3=("Saldo_Promedio_Mensual", "mean"),
            Atraso_Ultimos_3=("Dias_Atraso_Otras_Deudas", "mean")
        )
        .reset_index()
    )

    ultimo = historial.groupby("ID_Solicitud").tail(1)[
        [
            "ID_Solicitud",
            "Saldo_Promedio_Mensual",
            "Dias_Atraso_Otras_Deudas"
        ]
    ].rename(columns={
        "Saldo_Promedio_Mensual": "Saldo_Ultimo",
        "Dias_Atraso_Otras_Deudas": "Atraso_Ultimo"
    })

    # incorporamos las estadisticas a cada solicitud
    datos = sol.merge(
        generales, on="ID_Solicitud",
        how="left", validate="one_to_one"
    )

    datos = datos.merge(
        recientes, on="ID_Solicitud",
        how="left", validate="one_to_one"
    )

    datos = datos.merge(
        ultimo, on="ID_Solicitud",
        how="left", validate="one_to_one"
    )

    # calculamos las mismas relaciones financieras que en V7
    datos["Ratio_Monto_Ingreso"] = (
        datos["Monto_Solicitado"] /
        datos["Ingreso_Mensual"].clip(lower=1)
    )

    datos["Frecuencia_Atraso"] = (
        datos["Meses_Con_Atraso"] /
        datos["Meses_Observados"].clip(lower=1)
    )

    datos["Frecuencia_Atraso_30"] = (
        datos["Meses_Atraso_30"] /
        datos["Meses_Observados"].clip(lower=1)
    )

    datos["Monto_Sobre_Saldo"] = (
        datos["Monto_Solicitado"] /
        (datos["Saldo_Promedio"].clip(lower=0) + 100)
    )

    datos["Saldo_Sobre_Ingreso"] = (
        datos["Saldo_Promedio"] /
        datos["Ingreso_Mensual"].clip(lower=1)
    )

    datos["Ratio_Saldo_Reciente"] = (
        (datos["Saldo_Ultimos_3"] + 100) /
        (datos["Saldo_Promedio"].clip(lower=0) + 100)
    )

    datos["Diferencia_Atraso_Reciente"] = (
        datos["Atraso_Ultimos_3"] -
        datos["Atraso_Promedio"]
    )

    return datos


# preparamos entrenamiento y test sin modificar la V7
m21_train = preparar_datos_m21(train, bancario)
m21_test = preparar_datos_m21(test, bancario)

# verificamos que ambas versiones tengan las mismas solicitudes
assert np.array_equal(
    m21_train["ID_Solicitud"].to_numpy(),
    m19_train["ID_Solicitud"].to_numpy()
)

assert np.array_equal(
    m21_test["ID_Solicitud"].to_numpy(),
    m19_test["ID_Solicitud"].to_numpy()
)

assert not (set(m21_columnas) - set(m21_train.columns))
assert not (set(m21_columnas) - set(m21_test.columns))

print("DATOS PREPARADOS")
print("Train:", m21_train.shape)
print("Test:", m21_test.shape)

print("\nMESES DE HISTORIAL DISPONIBLES")
print("V7 train:", round(m19_train["Meses_Observados"].mean(), 2))
print("M21 train:", round(m21_train["Meses_Observados"].mean(), 2))
print("V7 test:", round(m19_test["Meses_Observados"].mean(), 2))
print("M21 test:", round(m21_test["Meses_Observados"].mean(), 2))


In [ ]:

# evaluamos la ganancia real de una seleccion de prestamos
def evaluar_cartera_m21(datos, probabilidades, presupuesto):
    cartera = datos[
        [
            "ID_Solicitud", "Monto_Solicitado",
            "Tasa_Interes", "Estado_Final"
        ]
    ].copy().reset_index(drop=True)

    cartera["Prob_Pago"] = np.asarray(probabilidades)

    cartera["EV_Dolar"] = (
        cartera["Prob_Pago"] *
        (1 + cartera["Tasa_Interes"]) - 1
    )

    cartera = cartera.sort_values(
        ["EV_Dolar", "ID_Solicitud"],
        ascending=[False, True],
        kind="mergesort"
    ).reset_index(drop=True)

    cartera["Aprobar"] = 0
    capital = 0.0

    # aprobamos prestamos sin superar el presupuesto
    for i, fila in enumerate(cartera.itertuples(index=False)):
        if fila.EV_Dolar <= 0:
            break

        if capital + fila.Monto_Solicitado <= presupuesto:
            cartera.loc[i, "Aprobar"] = 1
            capital += fila.Monto_Solicitado

    aprobados = cartera[cartera["Aprobar"] == 1]

    ganancia = np.where(
        aprobados["Estado_Final"] == 1,
        aprobados["Monto_Solicitado"] * aprobados["Tasa_Interes"],
        -aprobados["Monto_Solicitado"]
    ).sum()

    return {
        "Ganancia": float(ganancia),
        "ROI_%": 100 * ganancia / capital if capital else 0,
        "Capital": capital,
        "Aprobados": len(aprobados),
        "Defaults": int((aprobados["Estado_Final"] == 0).sum()),
        "Defaults_Esperados": float(
            (1 - aprobados["Prob_Pago"]).sum()
        ),
        "Uso_Presupuesto_%": (
            100 * capital / presupuesto if presupuesto else 0
        )
    }


m21_resultados = []
m21_predicciones = {}

# recorremos los siete meses de validacion
for mes in m21_meses:
    inicio = pd.Timestamp(mes + "-01")
    fin = inicio + pd.offsets.MonthBegin(1)

    anteriores = m19_train["Fecha_dt"] < inicio

    entrenamiento_mask = (
        anteriores &
        (m19_train["Monto_Solicitado"] <= m19_max_test)
    )

    usuarios_vistos = set(
        m19_train.loc[anteriores, "ID_Usuario"]
    )

    validacion_mask = (
        (m19_train["Fecha_dt"] >= inicio) &
        (m19_train["Fecha_dt"] < fin) &
        (m19_train["Monto_Solicitado"] <= m19_max_test) &
        (~m19_train["ID_Usuario"].isin(usuarios_vistos))
    )

    validacion_base = m19_train.loc[validacion_mask].copy()

    assert len(validacion_base) > 0, f"Sin validacion: {mes}"

    presupuesto = (
        validacion_base["Monto_Solicitado"].sum() *
        m19_proporcion
    )

    m21_predicciones[mes] = {
        "Datos": validacion_base.reset_index(drop=True),
        "Probabilidades": {}
    }

    # entrenamos las dos versiones con las mismas solicitudes
    for nombre, datos in [
        ("V7", m19_train),
        ("Ventana 12 meses", m21_train)
    ]:
        entrenamiento = datos.loc[entrenamiento_mask]
        validacion = datos.loc[validacion_mask]

        modelo = crear_logistica_m19(
            m21_columnas, C=0.1
        )

        modelo.fit(
            entrenamiento[m21_columnas],
            entrenamiento["Estado_Final"]
        )

        clases = list(
            modelo.named_steps["modelo"].classes_
        )

        probabilidades = modelo.predict_proba(
            validacion[m21_columnas]
        )[:, clases.index(1)]

        metricas = evaluar_cartera_m21(
            validacion, probabilidades, presupuesto
        )

        auc = roc_auc_score(
            validacion["Estado_Final"],
            probabilidades
        ) if validacion["Estado_Final"].nunique() == 2 else np.nan

        m21_resultados.append({
            "Mes": mes,
            "Version": nombre,
            "Candidatos": len(validacion),
            "Presupuesto": presupuesto,
            "AUC": auc,
            **metricas
        })

        m21_predicciones[mes]["Probabilidades"][nombre] = (
            probabilidades
        )

    print("Terminado:", mes)

m21_tabla = pd.DataFrame(m21_resultados)

print("\nValidacion temporal completada")
print("Evaluaciones:", len(m21_tabla))


In [ ]:

# comparamos la ganancia de ambas versiones por mes
m21_ganancias = m21_tabla.pivot(
    index="Mes",
    columns="Version",
    values="Ganancia"
)

m21_ganancias["Diferencia"] = (
    m21_ganancias["Ventana 12 meses"] -
    m21_ganancias["V7"]
)

print("GANANCIAS POR MES")
display(m21_ganancias.round(2))

# comparamos el retorno de inversion
print("\nROI POR MES")

display(
    m21_tabla.pivot(
        index="Mes",
        columns="Version",
        values="ROI_%"
    ).round(3)
)

# resumimos las metricas de las dos versiones
m21_resumen = m21_tabla.groupby("Version").agg(
    Ganancia_Promedio=("Ganancia", "mean"),
    ROI_Promedio=("ROI_%", "mean"),
    Peor_ROI=("ROI_%", "min"),
    AUC_Promedio=("AUC", "mean"),
    Defaults_Promedio=("Defaults", "mean"),
    Defaults_Esperados=("Defaults_Esperados", "mean"),
    Aprobados_Promedio=("Aprobados", "mean"),
    Uso_Presupuesto=("Uso_Presupuesto_%", "mean")
)

print("\nRESULTADOS GENERALES")
display(m21_resumen.round(3))

# comprobamos que reproducimos la validacion original de V7
control = m21_tabla[
    (m21_tabla["Version"] == "V7") &
    (m21_tabla["Mes"].isin(["2025-10", "2025-11", "2025-12"]))
]["Ganancia"].mean()

print("\nCONTROL DE REPRODUCCION")
print("V7 octubre-diciembre:", round(control, 2))
print("Resultado esperado:", 234026.64)

if abs(control - 234026.64) > 1:
    print("ATENCION: la validacion no reproduce el Modelo 20.")
    print("Revisar antes de sacar conclusiones.")

# contamos en cuantos meses gano cada version
meses_ganados = int(
    (m21_ganancias["Diferencia"] > 0).sum()
)

print("\nCONCLUSION INICIAL")
print("Meses ganados por ventana 12:", meses_ganados, "de 7")
print(
    "Diferencia promedio:",
    round(m21_ganancias["Diferencia"].mean(), 2)
)

if meses_ganados >= 5 and m21_ganancias["Diferencia"].mean() > 0:
    print("Resultado favorable. Falta comprobar estabilidad.")
else:
    print("No hay evidencia suficiente para reemplazar V7.")


In [ ]:

# dividimos los montos del test en grupos
m21_montos_test = m19_test["Monto_Solicitado"].to_numpy(float)

m21_cortes = np.unique(
    np.quantile(m21_montos_test, [0.20, 0.40, 0.60, 0.80])
)

m21_grupos_test = np.digitize(m21_montos_test, m21_cortes)

m21_proporciones = (
    np.bincount(
        m21_grupos_test,
        minlength=len(m21_cortes) + 1
    ) / len(m21_montos_test)
)

m21_generador = np.random.default_rng(42)
m21_repeticiones = 150

m21_bootstrap = []
m21_cobertura = []

# repetimos el muestreo usando los mismos datos para ambos modelos
for mes in m21_meses:
    datos = m21_predicciones[mes]["Datos"]

    montos = datos["Monto_Solicitado"].to_numpy(float)
    tasas = datos["Tasa_Interes"].to_numpy(float)
    pagos = datos["Estado_Final"].to_numpy()

    grupos = np.digitize(montos, m21_cortes)

    indices_grupo = [
        np.flatnonzero(grupos == g)
        for g in range(len(m21_proporciones))
    ]

    cantidades = [len(indices) for indices in indices_grupo]

    m21_cobertura.append({
        "Mes": mes,
        **{
            f"Grupo_{i+1}": cantidad
            for i, cantidad in enumerate(cantidades)
        }
    })

    # si falta un rango de monto, no podemos reproducir el test
    if any(
        cantidad == 0 and proporcion > 0
        for cantidad, proporcion in zip(
            cantidades, m21_proporciones
        )
    ):
        print(mes, "- bootstrap omitido: faltan rangos de monto")
        continue

    # distribuimos la muestra segun las proporciones del test
    esperados = m21_proporciones * len(datos)
    tamanos = np.floor(esperados).astype(int)

    faltantes = len(datos) - tamanos.sum()

    orden_resto = np.argsort(
        -(esperados - tamanos)
    )

    for posicion in orden_resto[:faltantes]:
        tamanos[posicion] += 1

    for repeticion in range(m21_repeticiones):
        muestra = np.concatenate([
            m21_generador.choice(
                indices_grupo[g],
                size=tamanos[g],
                replace=True
            )
            for g in range(len(tamanos))
            if tamanos[g] > 0
        ])

        muestra.sort()

        # mantenemos la misma proporcion presupuestaria
        presupuesto = (
            montos[muestra].sum() * m19_proporcion
        )

        muestra_datos = datos.iloc[muestra].reset_index(drop=True)

        for nombre in ["V7", "Ventana 12 meses"]:
            prob_original = (
                m21_predicciones[mes]["Probabilidades"][nombre]
            )

            metricas = evaluar_cartera_m21(
                muestra_datos,
                prob_original[muestra],
                presupuesto
            )

            m21_bootstrap.append({
                "Mes": mes,
                "Repeticion": repeticion,
                "Version": nombre,
                "Ganancia_Equiv_2M": (
                    metricas["Ganancia"] * 2_000_000 / presupuesto
                    if presupuesto else 0
                ),
                "ROI_%": metricas["ROI_%"]
            })

    print("Bootstrap terminado:", mes)

print("\nCOBERTURA DE MONTOS POR MES")
display(pd.DataFrame(m21_cobertura))

if m21_bootstrap:
    m21_boot_tabla = pd.DataFrame(m21_bootstrap)

    # comparamos los dos modelos sobre exactamente las mismas muestras
    m21_pareado = m21_boot_tabla.pivot(
        index=["Mes", "Repeticion"],
        columns="Version",
        values="Ganancia_Equiv_2M"
    )

    m21_pareado["Diferencia"] = (
        m21_pareado["Ventana 12 meses"] -
        m21_pareado["V7"]
    )

    m21_diferencias = m21_pareado.groupby("Mes")[
        "Diferencia"
    ].agg(
        Diferencia_Promedio="mean",
        Proporcion_Mejora=lambda x: (x > 0).mean()
    )

    print("\nRESULTADOS BOOTSTRAP POR MES")
    display(m21_diferencias.round(3))

    print("\nRESULTADOS GENERALES BOOTSTRAP")
    display(
        m21_boot_tabla.groupby("Version").agg(
            Ganancia_Equiv_2M=("Ganancia_Equiv_2M", "mean"),
            ROI_Promedio=("ROI_%", "mean")
        ).round(2)
    )

    print("\nESTABILIDAD DE LA MEJORA")
    print(
        "Meses evaluados:",
        len(m21_diferencias), "de 7"
    )
    print(
        "Porcentaje de muestras con mejora:",
        round(100 * (m21_pareado["Diferencia"] > 0).mean(), 2)
    )
else:
    print("No hay suficiente cobertura para realizar el bootstrap.")
